In [1]:
import os
import numpy as np
from PIL import Image as PILImage

# --- 配置 ---
IMG_DIR = './imgs'
SAVE_DIR = './imgs_preprocessed'
TARGET_SIZE = (700, 500)  # (W, H)
# 这里的 scale 需要和你的 DPU Encoder 输入 scale 一致 (2^-6 = 0.015625)
ENC_IN_SCALE = 0.015625 

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

files = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

print(f"开始预处理 {len(files)} 张图片...")

for f in files:
    # 1. 加载与缩放 (与训练/原始逻辑保持一致)
    img_pil = PILImage.open(os.path.join(IMG_DIR, f)).convert('RGB')
    img_small = img_pil.resize(TARGET_SIZE, PILImage.BILINEAR)
    
    # 2. 归一化到 [-1, 1]
    img_arr = np.asarray(img_small, dtype=np.float32)
    input_fp32 = (img_arr / 255.0 - 0.5) / 0.5
    
    # 3. 量化为 int8 (DPU 直接输入格式)
    input_int8 = np.clip(np.round(input_fp32 / ENC_IN_SCALE), -128, 127).astype(np.int8)
    
    # 4. 保存为 numpy 格式，读取速度极快
    save_path = os.path.join(SAVE_DIR, f + '.npy')
    np.save(save_path, input_int8)

print(f"✅ 处理完成，数据保存在: {SAVE_DIR}")

开始预处理 10 张图片...
✅ 处理完成，数据保存在: ./imgs_preprocessed
